# M3 — Tiền Xử Lý, Gán Nhãn & Trích Xuất Đặc Trưng (Colab + Drive)

Chạy **sau** khi M1 (crawl arXiv + Semantic Scholar) và M2 (parsing TeX/BibTeX) đã hoàn thành.

Quy trình: GitHub branch → Drive → đọc dữ liệu M1/M2 → gán nhãn thủ công + tự động → trích xuất 14 đặc trưng → chia split → xuất Parquet.

**Cần push code Python mới nhất lên GitHub branch trước khi chạy notebook.**

## SET UP — Clone code từ branch GitHub và cài thư viện

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/tuandatedu/arXir_project.git"
BRANCH = "M3-matching"
PROJECT = Path("/content/project")

if not (PROJECT / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "src/requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_m3_matching.py"], check=True)
print("✓ Repo ready:", PROJECT, "|", BRANCH)

## SET UP — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1 — Cấu hình đường dẫn

In [ ]:
RANGE_ID = 1

In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path

M1_REFS_JSON = Path(f"/content/drive/MyDrive/references/range_{RANGE_ID}/tables/references.json")
M2_BIBTEX_JSONL = Path(f"/content/drive/MyDrive/M2_processed/range_{RANGE_ID}/tables/bibtex.jsonl")

MY_M3_DIR = Path(f"/content/drive/MyDrive/M3_processed/range_{RANGE_ID}")
TABLES_DIR = MY_M3_DIR / "tables"
LABELS_DIR = MY_M3_DIR / "labels"
PREPARED_DATASET_DIR = MY_M3_DIR / "prepared_dataset"

for path in (TABLES_DIR, LABELS_DIR, PREPARED_DATASET_DIR):
    path.mkdir(parents=True, exist_ok=True)

assert M1_REFS_JSON.exists(), f"Không tìm thấy {M1_REFS_JSON}"
assert M2_BIBTEX_JSONL.exists(), f"Không tìm thấy {M2_BIBTEX_JSONL}"

RUN_ID = os.environ.get("M3_RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))

from src.common import config
import src.common.metrics as metrics_module
config.WORKING_DIR = MY_M3_DIR
config.TABLES_DIR = TABLES_DIR
metrics_module.TABLES_DIR = TABLES_DIR

print(f"Range: {RANGE_ID} | RUN_ID: {RUN_ID}")

## 2 — Nạp dữ liệu (M1 + M2) → JSON

In [ ]:
import json

def load_jsonl(file_path: Path) -> list:
    records = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def save_json(records: list, output_path: Path):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"✓ {output_path.name}: {len(records):,} records")

with open(M1_REFS_JSON, "r", encoding="utf-8") as f:
    references = json.load(f)
save_json(references, TABLES_DIR / "references.json")

bibtex = load_jsonl(M2_BIBTEX_JSONL)
save_json(bibtex, TABLES_DIR / "bibtex.json")

print(f"\n✓ Hoàn tát: {len(bibtex):,} BibTeX, {len(references):,} References")

## 3 — Gán nhãn thủ công (CSV → JSON)

In [ ]:
import pandas as pd
from src.m3_matching.manual_tool import (
    export_manual_template,
    find_candidate_manual_papers,
    import_manual_csv,
    summarize_team_labels,
)

candidates = find_candidate_manual_papers(
    bibtex_path=TABLES_DIR / "bibtex.json",
    references_path=TABLES_DIR / "references.json",
)
print(f"Tìm thấy {len(candidates)} bài đủ điều kiện")
assert len(candidates) >= 5, f"Chỉ có {len(candidates)} bài (cần ≥5)"

MANUAL_DIR = LABELS_DIR / "manual"
MANUAL_DIR.mkdir(parents=True, exist_ok=True)
selected_5 = [pid for pid, _, _ in candidates[:5]]

for pid in selected_5:
    export_manual_template(
        pid, MANUAL_DIR / f"{pid}.csv",
        bibtex_path=TABLES_DIR / "bibtex.json",
        references_path=TABLES_DIR / "references.json",
    )

print(f"\n✓ Đã xuất {len(selected_5)} CSV tại: {MANUAL_DIR}")
print(f"   → Mở Excel, điền groundtruth_arxiv_id + is_verified=YES")
print(f"   → Upload lên Drive, chạy cell tiếp theo")

df_preview = pd.read_csv(MANUAL_DIR / f"{selected_5[0]}.csv")
display(df_preview[["citation_key", "bib_title", "suggested_ref_arxiv_id", "suggested_score", "groundtruth_arxiv_id"]].head(5))

In [ ]:
for pid in selected_5:
    csv_path = MANUAL_DIR / f"{pid}.csv"
    if csv_path.exists():
        import_manual_csv(csv_path, pid, labels_dir=LABELS_DIR)
        print(f"✓ Imported {pid}")

summary = summarize_team_labels(labels_dir=LABELS_DIR)
print(f"\nManual papers: {summary['n_manual_papers']} (cần ≥5)")
print(f"Manual entries: {summary['n_manual_entries']} (cần ≥100)")

## 4 — Pipeline M3 (14 features + split)

In [ ]:
from src.m3_matching.prepare_dataset import run_preparation_pipeline

summary = run_preparation_pipeline(
    tables_dir=TABLES_DIR,
    labels_dir=LABELS_DIR,
    k=2,
    seed=42,
    demo=False,
)

print(f"\n=== SPLITS ===")
for split in ["train", "valid", "test"]:
    info = summary["splits"][split]
    print(f"{split:5s}: {info['n_pairs']:,} pairs (pos={info['n_positives']}, neg={info['n_negatives']})")

print(f"\n✓ Files in prepared_dataset/:")
for f in summary["exported_files"]:
    p = Path(f)
    if p.exists():
        size_mb = p.stat().st_size / 1024 / 1024
        print(f"  {p.name}: {size_mb:.1f} MB")

## 5 — Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

train_df = pd.read_parquet(PREPARED_DATASET_DIR / "train_pairs.parquet")
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(data=train_df, x="title_token_sort", hue="label", bins=30, kde=True, ax=axes[0], palette={0: "#e74c3c", 1: "#2ecc71"}, common_norm=False, stat="density")
axes[0].set_title("Title Similarity (Token Sort Ratio)")
axes[0].legend(title="Label", labels=["Negative (0)", "Positive (1)"])

sns.histplot(data=train_df, x="author_jaccard", hue="label", bins=20, ax=axes[1], palette={0: "#e74c3c", 1: "#2ecc71"}, common_norm=False, stat="density")
axes[1].set_title("Author Overlap (Jaccard)")
axes[1].legend(title="Label", labels=["Negative (0)", "Positive (1)"])

year_df = train_df[train_df["year_diff"] < 20]
sns.boxplot(data=year_df, x="label", y="year_diff", ax=axes[2], palette={0: "#e74c3c", 1: "#2ecc71"})
axes[2].set_title("Year Difference")
axes[2].set_xticklabels(["Negative (0)", "Positive (1)"])

plt.tight_layout()
poster_fig_path = PREPARED_DATASET_DIR / "poster_feature_distribution.png"
plt.savefig(poster_fig_path, dpi=300)
plt.show()
print(f"✓ Saved: {poster_fig_path}")

## 6 — Deliverables

In [ ]:
print(f"\n{'='*70}")
print(f"MILESTONE 3 — COMPLETE")
print(f"{'='*70}")

print(f"\nLocation: {MY_M3_DIR}")
print(f"\n📊 Structure:")
print(f"  tables/")
print(f"    ├── bibtex.json")
print(f"    └── references.json")
print(f"  labels/")
print(f"    ├── manual/")
print(f"    │   ├── 2310.12345.csv  (5 files - edit in Excel)")
print(f"    │   ├── 2310.12346.csv")
print(f"    │   ├── ... .csv")
print(f"    │   ├── 2310.12345.json (imported)")
print(f"    │   ├── 2310.12346.json")
print(f"    │   └── ... .json")
print(f"    └── automatic/")
print(f"        ├── 2310.12345.json")
print(f"        └── ... .json")
print(f"  prepared_dataset/             (🚀 for Person 2)")
print(f"    ├── train_pairs.parquet")
print(f"    ├── valid_pairs.parquet")
print(f"    ├── test_pairs.parquet")
print(f"    ├── dataset_pairs.parquet")
print(f"    └── poster_feature_distribution.png")

print(f"\n✅ Person 1 deliverables:")
print(f"  [✓] Data cleaning: bibtex.json, references.json")
print(f"  [✓] Manual labeling: 5 CSV + JSON")
print(f"  [✓] Auto labeling: JSON")
print(f"  [✓] Feature engineering: 14 features")
print(f"  [✓] Dataset creation: pairs with label")
print(f"  [✓] Train/valid/test split")

print(f"\n🚀 Person 2 receives:")
print(f"  • prepared_dataset/ folder")
print(f"  • 4 Parquet files + visualization")
print(f"  • Ready for model training")

print(f"\nRUN_ID: {RUN_ID}")
print(f"{'='*70}")